============================================================
# EMBEDDING REPRODUCIBILITY + SGA COMPARISON
============================================================

Kernel: SSL-PL

Generated using ChatGPT

PURPOSE
-------
Investigate why embeddings generated by our morphology
classification pipeline differ from the official SGA2025
SSL embeddings, despite using the same ResNet50 checkpoint.

We test:

  TEST 1 — Reproducibility with current augmentations
            Do repeated passes over the same galaxies produce
            different images/embeddings?

  TEST 2 — Reproducibility without augmentations
            Are repeated passes deterministic when all
            augmentations are disabled?

  TEST 3 — Our deterministic embeddings vs official SGA
            Do embeddings generated from the same images using
            the same checkpoint exactly reproduce SGA's embeddings?

  TEST 4 — Checkpoint identity
            Are the checkpoint files actually identical?

  TEST 5 — Input pipeline
            Does our no-augmentation dataset produce exactly
            the raw HDF5 image supplied to the official pipeline?

  TEST 6 — Model-loading implementation
            Does the official pipeline's direct torchvision
            model-loading approach produce the same output as
            Moco_v2.load_from_checkpoint()?

IMPORTANT
---------
All reproducibility tests use the same 100 galaxies from the
beginning of chunk 0007.

Results are recorded below each numbered test.

In [2]:
# ============================================================
# Imports
# ============================================================

import h5py
import numpy as np
import torch
import torchvision

from pathlib import Path
from tqdm import tqdm

from ssl_legacysurvey.data_loaders import datamodules
from ssl_legacysurvey.moco.moco2_module import Moco_v2

In [3]:
# ============================================================
# Configuration
# ============================================================

CHUNK_PATH = Path(
    "/global/cfs/cdirs/desicollab/users/ioannis"
    "/SGA/2025/ssl/ssl-cutouts-dr11-south-chunk0007.hdf5"
)

OFFICIAL_EMBEDDINGS_PATH = Path(
    "/global/cfs/cdirs/desicollab/users/ioannis"
    "/SGA/2025/ssl/ssl-embeddings-dr11-south.hdf5"
)

TEST_DATA_PATH = Path(
    "/pscratch/sd/q/qshimp/SGA2025-data/"
    "embedding_comparison_100.npz"
)

OFFICIAL_CHECKPOINT = Path(
    "/global/cfs/cdirs/desicollab/users/ioannis"
    "/ssl-legacysurvey/resnet50.ckpt"
)

MY_CHECKPOINT = Path(
    "/global/homes/q/qshimp/morphology_classification"
    "/Binary-classifier/resnet50.ckpt"
)

N_TEST = 100
EMBEDDING_BATCH_SIZE = 128

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print(f"Test galaxies:       {N_TEST}")
print(f"Chunk:                {CHUNK_PATH.name}")
print(f"Official embeddings:  {OFFICIAL_EMBEDDINGS_PATH.name}")
print(f"Official checkpoint:  {OFFICIAL_CHECKPOINT}")
print(f"Our checkpoint:       {MY_CHECKPOINT}")

EXPERIMENT CONFIGURATION
Test galaxies:       100
Chunk:                ssl-cutouts-dr11-south-chunk0007.hdf5
Official embeddings:  ssl-embeddings-dr11-south.hdf5
Official checkpoint:  /global/cfs/cdirs/desicollab/users/ioannis/ssl-legacysurvey/resnet50.ckpt
Our checkpoint:       /global/homes/q/qshimp/morphology_classification/Binary-classifier/resnet50.ckpt


In [4]:
# ============================================================
# Load fixed 100-galaxy test set
# ============================================================

test_data = np.load(TEST_DATA_PATH)

test_ids = test_data["sgaid"]
official_embeddings = test_data["official_embeddings"]

print(f"Test galaxies:       {len(test_ids)}")
print(f"Official embeddings: {official_embeddings.shape}")

if len(test_ids) != N_TEST:
    raise ValueError(
        f"Expected {N_TEST} test galaxies, found {len(test_ids)}"
    )

with h5py.File(CHUNK_PATH, "r") as f:
    chunk_ids = f["sgaid"][:N_TEST]

if not np.array_equal(test_ids, chunk_ids):
    raise ValueError(
        "Test SGAIDs do not match the first 100 galaxies "
        "in chunk 0007!"
    )

print("Chunk SGAIDs verified.")

Test galaxies:       100
Official embeddings: (100, 2048)
Chunk SGAIDs verified.


In [5]:
# ============================================================
# Helper functions
# ============================================================

def generate_embeddings(images):
    """
    Generate 2048-dimensional ResNet50 encoder_q embeddings.

    The model is placed in evaluation mode and inference is performed
    without gradients. No additional augmentation occurs here; all
    image transformations happen before this function is called.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_backbone = backbone.to(device)
    model_backbone.eval()

    embeddings = []

    with torch.no_grad():

        for start in tqdm(range(0, len(images), EMBEDDING_BATCH_SIZE), desc="Generating embeddings"):

            end = min(start + EMBEDDING_BATCH_SIZE, len(images))

            batch = images[start:end].to(device)

            output = model_backbone(batch)

            embeddings.append(output.cpu())

    return torch.cat(embeddings)

def compare_tensors(a, b, name):

    if a.shape != b.shape:
        raise ValueError(
            f"{name}: shape mismatch: "
            f"{a.shape} vs {b.shape}"
        )

    difference = a - b

    max_diff = torch.max(torch.abs(difference)).item()
    mean_diff = torch.mean(torch.abs(difference)).item()
    rmse = torch.sqrt(torch.mean(difference ** 2)).item()

    print()
    print(name)
    print("-" * len(name))

    print(f"Shape:                 {a.shape}")
    print(f"Exactly equal:         {torch.equal(a, b)}")
    print(f"Max absolute diff:     {max_diff:.10e}")
    print(f"Mean absolute diff:    {mean_diff:.10e}")
    print(f"RMSE:                  {rmse:.10e}")

    return difference

def build_dataset(augmentations, label_path):

    params = {
        "augmentations": augmentations,
        "npix_out": 152,
        "jitter_lim": 0,
        "verbose": False,
    }

    transform = datamodules.DecalsTransforms(
        augmentations,
        {
            "npix_out": 152,
            "jitter_lim": 0,
        }
    )

    return datamodules.DecalsDataset(
        str(CHUNK_PATH),
        str(label_path),
        transform,
        params,
    )

def process_test_images(dataset):

    images = []

    for i in tqdm(
        range(len(dataset)),
        desc="Processing images"
    ):
        image, _ = dataset[i]
        images.append(image)

    return torch.stack(images)

In [6]:
# ============================================================
# Create label file for DecalsDataset
# ============================================================

test_indices = np.arange(N_TEST)

label_array = np.column_stack([
    test_indices,
    np.zeros(N_TEST, dtype=np.int64)
]).astype(np.int64)

LABEL_PATH = Path("/tmp/embedding_comparison_100_labels.npy")

np.save(LABEL_PATH, label_array)

print(f"Label file: {LABEL_PATH}")
print(f"Label array shape: {label_array.shape}")

Label file: /tmp/embedding_comparison_100_labels.npy
Label array shape: (100, 2)


In [7]:
# ============================================================
# MODEL — Load ResNet50 encoder_q
# ============================================================

print("=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = Moco_v2.load_from_checkpoint(
    checkpoint_path=MY_CHECKPOINT
)

backbone = model.encoder_q

# The official SSL embeddings are the 2048-dimensional
# avgpool features, so remove the MoCo classification head.
backbone.fc = torch.nn.Identity()

backbone.eval()

print(f"Checkpoint: {MY_CHECKPOINT}")
print("Backbone:   ResNet50 encoder_q")
print("Output:     2048-dimensional embeddings")
print("Mode:       eval()")
print("Model loaded successfully.")

LOADING MODEL
Checkpoint: /global/homes/q/qshimp/morphology_classification/Binary-classifier/resnet50.ckpt
Backbone:   ResNet50 encoder_q
Output:     2048-dimensional embeddings
Mode:       eval()
Model loaded successfully.


In [10]:
# ============================================================
# TEST 1 — REPRODUCIBILITY WITH CURRENT AUGMENTATIONS
# ============================================================
#
# QUESTION
# --------
# If we process the exact same 100 galaxies twice using the
# morphology-classification pipeline's current augmentations,
# do we obtain identical images and embeddings?
#
# Current augmentation configuration:
#     rrjc = RandomRotate + JitterCrop
#
# INTERPRETATION
# --------------
# If images differ, the preprocessing pipeline is stochastic.
#
# If embeddings differ, those image differences propagate through
# the deterministic neural network into different embeddings.
# ============================================================

print()
print("=" * 60)
print("TEST 1 — CURRENT AUGMENTATIONS (rrjc)")
print("=" * 60)

dataset1 = build_dataset(
    "rrjc",
    LABEL_PATH
)

dataset2 = build_dataset(
    "rrjc",
    LABEL_PATH
)

print(
    "Augmentations:",
    dataset1.transforms.augmentation_names
)


images1 = process_test_images(dataset1)
images2 = process_test_images(dataset2)


compare_tensors(
    images1,
    images2,
    "TEST 1 — IMAGE COMPARISON"
)


embeddings1 = generate_embeddings(images1)
embeddings2 = generate_embeddings(images2)


compare_tensors(
    embeddings1,
    embeddings2,
    "TEST 1 — EMBEDDING COMPARISON"
)

# ============================================================
# TEST 1 — RESULT
# ============================================================

print("""
RESULT:
The current rrjc preprocessing is stochastic.

The same galaxies do NOT produce identical processed images,
and therefore do NOT produce identical embeddings.

This establishes that the old morphology-classification
embedding pipeline can produce different embeddings for the
same galaxy on different passes when rrjc is enabled.
""")


TEST 1 — CURRENT AUGMENTATIONS (rrjc)
Augmentations: ['RandomRotate', 'JitterCrop']


Processing images: 100%|██████████| 100/100 [00:00<00:00, 104.52it/s]



TEST 1 — IMAGE COMPARISON
-------------------------
Shape:                 torch.Size([100, 3, 152, 152])
Exactly equal:         False
Max absolute diff:     4.0937690735e+01
Mean absolute diff:    1.8672375008e-02
RMSE:                  2.7922666073e-01


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 39.23it/s]



TEST 1 — EMBEDDING COMPARISON
-----------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         False
Max absolute diff:     7.5621967316e+00
Mean absolute diff:    4.1633054614e-02
RMSE:                  1.9315026700e-01

RESULT:
The current rrjc preprocessing is stochastic.

The same galaxies do NOT produce identical processed images,
and therefore do NOT produce identical embeddings.

This establishes that the old morphology-classification
embedding pipeline can produce different embeddings for the
same galaxy on different passes when rrjc is enabled.



In [11]:
# ============================================================
# TEST 2 — REPRODUCIBILITY WITHOUT AUGMENTATIONS
# ============================================================
#
# QUESTION
# --------
# If all augmentations are disabled, does processing the same
# galaxy twice produce exactly the same image and embedding?
#
# PURPOSE
# -------
# Determine whether the neural-network inference itself is
# stochastic, independent of image augmentation.
#
# ============================================================

print()
print("=" * 60)
print("TEST 2 — NO AUGMENTATIONS")
print("=" * 60)

dataset3 = build_dataset(
    "",
    LABEL_PATH
)

dataset4 = build_dataset(
    "",
    LABEL_PATH
)

print(
    "Augmentations:",
    dataset3.transforms.augmentation_names
)


images3 = process_test_images(dataset3)
images4 = process_test_images(dataset4)


compare_tensors(
    images3,
    images4,
    "TEST 2 — IMAGE COMPARISON"
)


embeddings3 = generate_embeddings(images3)
embeddings4 = generate_embeddings(images4)


compare_tensors(
    embeddings3,
    embeddings4,
    "TEST 2 — EMBEDDING COMPARISON"
)

# ============================================================
# TEST 2 — RESULT
# ============================================================

print("""
RESULT:
With augmentations disabled, the processed images are exactly
identical between repeated passes, and the resulting embeddings
are also exactly identical.

Therefore, the backbone inference is deterministic under this
setup.
""")


TEST 2 — NO AUGMENTATIONS
Augmentations: []


Processing images: 100%|██████████| 100/100 [00:00<00:00, 115.14it/s]



TEST 2 — IMAGE COMPARISON
-------------------------
Shape:                 torch.Size([100, 3, 152, 152])
Exactly equal:         True
Max absolute diff:     0.0000000000e+00
Mean absolute diff:    0.0000000000e+00
RMSE:                  0.0000000000e+00


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 39.56it/s]


TEST 2 — EMBEDDING COMPARISON
-----------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         True
Max absolute diff:     0.0000000000e+00
Mean absolute diff:    0.0000000000e+00
RMSE:                  0.0000000000e+00

RESULT:
With augmentations disabled, the processed images are exactly
identical between repeated passes, and the resulting embeddings
are also exactly identical.

Therefore, the backbone inference is deterministic under this
setup.



In [12]:
# ============================================================
# TEST 3 — OUR DETERMINISTIC EMBEDDINGS vs OFFICIAL SGA
# ============================================================
#
# QUESTION
# --------
# Using the same 100 galaxies, no augmentations, and the same
# checkpoint, do our generated 2048-dimensional embeddings
# exactly reproduce the official SGA embeddings?
#
# PURPOSE
# -------
# Determine whether differences between our embeddings and SGA's
# can be explained simply by stochastic augmentation.
#
# ============================================================

print()
print("=" * 60)
print("TEST 3 — OUR EMBEDDINGS vs OFFICIAL SGA")
print("=" * 60)


my_embeddings = embeddings3.numpy()


if my_embeddings.shape != official_embeddings.shape:

    raise ValueError(
        "Embedding shapes do not match:\n"
        f"  Mine: {my_embeddings.shape}\n"
        f"  SGA:  {official_embeddings.shape}"
    )


difference = (
    my_embeddings -
    official_embeddings
)


print()
print("DIRECT SUBTRACTION")
print("-------------------")

print(
    f"Maximum absolute difference: "
    f"{np.max(np.abs(difference)):.10e}"
)

print(
    f"Mean absolute difference:    "
    f"{np.mean(np.abs(difference)):.10e}"
)

print(
    f"RMSE:                        "
    f"{np.sqrt(np.mean(difference ** 2)):.10e}"
)

print()

print(
    "Exactly equal:",
    np.array_equal(
        my_embeddings,
        official_embeddings
    )
)

print(
    "Equal within tolerance:",
    np.allclose(
        my_embeddings,
        official_embeddings
    )
)

# ============================================================
# TEST 3 — RESULT
# ============================================================

print("""
RESULT:
The deterministic embeddings do NOT exactly reproduce the
official SGA embeddings.

Therefore, random augmentation is NOT sufficient to explain
the difference between our embeddings and the official SGA
embeddings.
""")


TEST 3 — OUR EMBEDDINGS vs OFFICIAL SGA

DIRECT SUBTRACTION
-------------------
Maximum absolute difference: 3.4503688812e+00
Mean absolute difference:    1.7472395673e-02
RMSE:                        8.5440747440e-02

Exactly equal: False
Equal within tolerance: False

RESULT:
The deterministic embeddings do NOT exactly reproduce the
official SGA embeddings.

Therefore, random augmentation is NOT sufficient to explain
the difference between our embeddings and the official SGA
embeddings.



In [13]:
# ============================================================
# TEST 4 — CHECKPOINT IDENTITY
# ============================================================
#
# QUESTION
# --------
# Are the checkpoint files used by our morphology classifier
# and the official SGA pipeline actually identical?
#
# ============================================================

official = torch.load(
    OFFICIAL_CHECKPOINT,
    map_location="cpu"
)

mine = torch.load(
    MY_CHECKPOINT,
    map_location="cpu"
)

official_state = official["state_dict"]
my_state = mine["state_dict"]

official_encoder = {
    k: v for k, v in official_state.items()
    if k.startswith("encoder_q.")
}

my_encoder = {
    k: v for k, v in my_state.items()
    if k.startswith("encoder_q.")
}

print("=" * 70)
print("TEST 4 — CHECKPOINT IDENTITY")
print("=" * 70)

print(f"Official encoder parameters: {len(official_encoder)}")
print(f"My encoder parameters:       {len(my_encoder)}")

print(
    "Same parameter names:",
    set(official_encoder.keys()) == set(my_encoder.keys())
)

different = 0
max_difference = 0.0

for key in official_encoder:

    official_param = official_encoder[key]
    my_param = my_encoder[key]

    if not torch.equal(official_param, my_param):

        different += 1

        difference = torch.max(
            torch.abs(official_param - my_param)
        ).item()

        max_difference = max(
            max_difference,
            difference
        )

print(f"Different tensors:           {different}")
print(f"Maximum weight difference:   {max_difference:.10e}")

print()
print(
    "CHECKPOINTS IDENTICAL:",
    different == 0
)

# ============================================================
# TEST 4 — RESULT
# ============================================================

print("""
RESULT:
The official and morphology-classification checkpoint files
contain identical model parameters.

Therefore, the checkpoint itself is not responsible for the
observed embedding differences.
""")

TEST 4 — CHECKPOINT IDENTITY
Official encoder parameters: 322
My encoder parameters:       322
Same parameter names: True
Different tensors:           0
Maximum weight difference:   0.0000000000e+00

CHECKPOINTS IDENTICAL: True

RESULT:
The official and morphology-classification checkpoint files
contain identical model parameters.

Therefore, the checkpoint itself is not responsible for the
observed embedding differences.



In [14]:
# ============================================================
# TEST 5 — RAW HDF5 INPUT vs OUR NO-AUGMENTATION INPUT
# ============================================================
#
# QUESTION
# --------
# Does our no-augmentation DecalsDataset provide exactly the
# same image tensor that the official embedding pipeline feeds
# into the ResNet50?
#
# ============================================================

with h5py.File(CHUNK_PATH, "r") as f:
    raw_images = f["images"][:N_TEST]

raw_images_torch = torch.from_numpy(raw_images)

noaug_dataset = build_dataset("", LABEL_PATH)
processed = process_test_images(noaug_dataset)

print("Raw HDF5 shape:", raw_images_torch.shape)
print("Our input shape:", processed.shape)

compare_tensors(
    processed,
    raw_images_torch,
    "TEST 5a — RAW HDF5 vs OUR INPUT"
)

print()
print("=" * 60)
print("TEST 5b — RAW HDF5 vs OUR INPUT: TRANSPOSE CHECK")
print("=" * 60)

# Transpose the spatial dimensions of the raw image:
# (N, C, H, W) -> (N, C, W, H)
raw_transposed = raw_images_torch.transpose(2, 3)

compare_tensors(
    processed,
    raw_transposed,
    "TEST 5b — OUR INPUT vs TRANSPOSED RAW HDF5"
)

print()
print("=" * 60)
print("TEST 5c — TRANSPOSED RAW vs DATASET EMBEDDINGS")
print("=" * 60)

raw_transposed_embeddings = generate_embeddings(
    raw_transposed
)

compare_tensors(
    embeddings3,
    raw_transposed_embeddings,
    "TEST 5c — EMBEDDING COMPARISON"
)

# ============================================================
# TEST 5 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 5 — SUMMARY")
print("=" * 60)

print("Question:")
print("Does DecalsDataset alter the raw HDF5 image values?")
print()

print("Result:")
print(f"  Raw shape:              {raw_images.shape}")
print(f"  Dataset input shape:    {processed.shape}")
print(f"  Raw dtype:              {raw_images.dtype}")
print(f"  Dataset dtype:          {processed.dtype}")
print()

print("Conclusion:")
print("  The dataset output is NOT numerically identical to the")
print("  raw HDF5 array because the image axes are transposed.")
print("  After applying the corresponding transpose, the tensors")
print("  are exactly equal.")
print()
print("  Therefore, the transformation performed by DecalsDataset")
print("  is accounted for by the axis transpose tested here.")

Processing images: 100%|██████████| 100/100 [00:00<00:00, 120.24it/s]


Raw HDF5 shape: torch.Size([100, 3, 152, 152])
Our input shape: torch.Size([100, 3, 152, 152])

TEST 5a — RAW HDF5 vs OUR INPUT
-------------------------------
Shape:                 torch.Size([100, 3, 152, 152])
Exactly equal:         False
Max absolute diff:     4.4536571503e+01
Mean absolute diff:    2.4340232834e-02
RMSE:                  3.4723451734e-01

TEST 5b — RAW HDF5 vs OUR INPUT: TRANSPOSE CHECK

TEST 5b — OUR INPUT vs TRANSPOSED RAW HDF5
------------------------------------------
Shape:                 torch.Size([100, 3, 152, 152])
Exactly equal:         True
Max absolute diff:     0.0000000000e+00
Mean absolute diff:    0.0000000000e+00
RMSE:                  0.0000000000e+00

TEST 5c — TRANSPOSED RAW vs DATASET EMBEDDINGS


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 38.75it/s]


TEST 5c — EMBEDDING COMPARISON
------------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         True
Max absolute diff:     0.0000000000e+00
Mean absolute diff:    0.0000000000e+00
RMSE:                  0.0000000000e+00

TEST 5 — SUMMARY
Question:
Does DecalsDataset alter the raw HDF5 image values?

Result:
  Raw shape:              (100, 3, 152, 152)
  Dataset input shape:    torch.Size([100, 3, 152, 152])
  Raw dtype:              float32
  Dataset dtype:          torch.float32

Conclusion:
  The dataset output is NOT numerically identical to the
  raw HDF5 array because the image axes are transposed.
  After applying the corresponding transpose, the tensors
  are exactly equal.

  Therefore, the transformation performed by DecalsDataset
  is accounted for by the axis transpose tested here.


In [15]:
# ============================================================
# TEST 6 — MODEL LOADING IMPLEMENTATION
# ============================================================
#
# QUESTION
# --------
# Does loading the checkpoint using the official pipeline's
# torchvision implementation produce the same backbone output
# as loading it using Moco_v2.load_from_checkpoint()?
#
# PURPOSE
# -------
# Isolate model-loading / architecture differences from
# preprocessing differences.
#
# ============================================================

def load_official_backbone(checkpoint_path, device):

    ckpt = torch.load(
        checkpoint_path,
        map_location=device,
    )

    state_dict = ckpt.get("state_dict", ckpt)

    backbone_state = {
        k[len("encoder_q."):]: v
        for k, v in state_dict.items()
        if k.startswith("encoder_q.")
    }

    backbone = torchvision.models.resnet50()
    backbone.fc = torch.nn.Identity()

    missing, unexpected = backbone.load_state_dict(
        backbone_state,
        strict=False
    )

    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    backbone = backbone.to(device).eval()

    return backbone

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

moco_backbone = Moco_v2.load_from_checkpoint(
    checkpoint_path=MY_CHECKPOINT
).encoder_q

moco_backbone.fc = torch.nn.Identity()
moco_backbone = moco_backbone.to(device).eval()

official_backbone = load_official_backbone(
    OFFICIAL_CHECKPOINT,
    device
)

with torch.no_grad():

    output_moco = moco_backbone(
        processed.to(device)
    ).cpu()

    output_official = official_backbone(
        processed.to(device)
    ).cpu()

compare_tensors(
    output_moco,
    output_official,
    "TEST 6 — MODEL OUTPUT COMPARISON"
)

# ============================================================
# TEST 6 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 6 — SUMMARY")
print("=" * 60)

print("Question:")
print("Does the method used to load the checkpoint change the")
print("2048-dimensional backbone embeddings?")
print()

print("Result:")
print(f"  Exactly equal:          {torch.equal(output_moco, output_official)}")
print()

print("Conclusion:")
print("  The Moco_v2 Lightning loader and the direct torchvision")
print("  ResNet50 loader produce exactly identical backbone outputs.")
print("  Model-loading implementation is therefore ruled out as")
print("  a source of the observed discrepancy.")

Missing keys: []
Unexpected keys: ['fc.0.weight', 'fc.0.bias', 'fc.2.weight', 'fc.2.bias']

TEST 6 — MODEL OUTPUT COMPARISON
--------------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         True
Max absolute diff:     0.0000000000e+00
Mean absolute diff:    0.0000000000e+00
RMSE:                  0.0000000000e+00

TEST 6 — SUMMARY
Question:
Does the method used to load the checkpoint change the
2048-dimensional backbone embeddings?

Result:
  Exactly equal:          True

Conclusion:
  The Moco_v2 Lightning loader and the direct torchvision
  ResNet50 loader produce exactly identical backbone outputs.
  Model-loading implementation is therefore ruled out as
  a source of the observed discrepancy.


In [16]:
# ============================================================
# TEST 7 — REPRODUCE OFFICIAL SGA EMBEDDINGS
# ============================================================

print()
print("=" * 60)
print("TEST 7 — REPRODUCE OFFICIAL SGA EMBEDDINGS")
print("=" * 60)

with torch.no_grad():

    reproduced_official = official_backbone(
        raw_images_torch.to(device)
    ).cpu()

compare_tensors(
    reproduced_official,
    torch.from_numpy(official_embeddings),
    "TEST 7 — REPRODUCED vs OFFICIAL EMBEDDINGS"
)

# ============================================================
# TEST 7 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 7 — SUMMARY")
print("=" * 60)

print("Question:")
print("Can our pipeline reproduce the official SGA embeddings?")
print()

print("Result:")
print(f"  Exactly equal:          {np.array_equal(my_embeddings, official_embeddings)}")
print(f"  Allclose:               {np.allclose(my_embeddings, official_embeddings)}")
print(f"  Max absolute diff:      {np.max(np.abs(difference)):.10e}")
print(f"  Mean absolute diff:     {np.mean(np.abs(difference)):.10e}")
print(f"  RMSE:                   {np.sqrt(np.mean(difference ** 2)):.10e}")
print()

print("Conclusion:")
print("  The reproduced embeddings are not numerically identical")
print("  to the official embeddings.")
print("  The source of this discrepancy remains unresolved.")


TEST 7 — REPRODUCE OFFICIAL SGA EMBEDDINGS

TEST 7 — REPRODUCED vs OFFICIAL EMBEDDINGS
------------------------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         False
Max absolute diff:     1.3046264648e-02
Mean absolute diff:    7.3706658441e-05
RMSE:                  3.1403396861e-04

TEST 7 — SUMMARY
Question:
Can our pipeline reproduce the official SGA embeddings?

Result:
  Exactly equal:          False
  Allclose:               False
  Max absolute diff:      3.4503688812e+00
  Mean absolute diff:     1.7472395673e-02
  RMSE:                   8.5440747440e-02

Conclusion:
  The reproduced embeddings are not numerically identical
  to the official embeddings.
  The source of this discrepancy remains unresolved.


In [17]:
# ============================================================
# TEST 8 — CPU vs GPU INFERENCE
# ============================================================
#
# QUESTION
# --------
# Can differences between CPU and GPU inference explain the
# discrepancy between our reproduced embeddings and the
# official SGA embeddings?
#
# PURPOSE
# -------
# The official embeddings may have been generated on a
# different device from the one used for reproduction.
#
# We therefore run the EXACT SAME:
#
#     input tensor
#     checkpoint
#     ResNet50 architecture
#
# on CPU and GPU and compare the resulting embeddings.
#
# INTERPRETATION
# --------------
# If CPU vs GPU differences are tiny compared with the
# discrepancy in TEST 7, hardware/device numerical differences
# can effectively be ruled out.
#
# IMPORTANT
# ---------
# This test does NOT compare different preprocessing pipelines.
# Both devices receive the exact same tensor.
#
# ============================================================

print()
print("=" * 60)
print("TEST 8 — CPU vs GPU INFERENCE")
print("=" * 60)

# ------------------------------------------------------------
# Use the deterministic no-augmentation input from TEST 5.
# ------------------------------------------------------------

test_input = processed

print("Input shape:", test_input.shape)
print("Input dtype:", test_input.dtype)


# ------------------------------------------------------------
# CPU
# ------------------------------------------------------------

print()
print("Running CPU inference...")

cpu_model = Moco_v2.load_from_checkpoint(
    checkpoint_path=MY_CHECKPOINT
)

cpu_backbone = cpu_model.encoder_q
cpu_backbone.fc = torch.nn.Identity()
cpu_backbone = cpu_backbone.to("cpu").eval()

with torch.no_grad():

    cpu_embeddings = cpu_backbone(
        test_input
    ).cpu()


# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

if torch.cuda.is_available():

    print("Running GPU inference...")

    gpu_model = Moco_v2.load_from_checkpoint(
        checkpoint_path=MY_CHECKPOINT
    )

    gpu_backbone = gpu_model.encoder_q
    gpu_backbone.fc = torch.nn.Identity()
    gpu_backbone = gpu_backbone.to("cuda").eval()

    with torch.no_grad():

        gpu_embeddings = gpu_backbone(
            test_input.to("cuda")
        ).cpu()

    compare_tensors(
        cpu_embeddings,
        gpu_embeddings,
        "TEST 8 — CPU vs GPU EMBEDDING COMPARISON"
    )

else:

    print()
    print("CUDA is not available.")
    print("TEST 8 cannot perform a CPU vs GPU comparison.")

# ============================================================
# TEST 8 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 8 — SUMMARY")
print("=" * 60)

print("Question:")
print("Can CPU vs GPU inference explain the embedding discrepancy?")
print()

if torch.cuda.is_available():

    print("Result:")
    print("  CPU and GPU inference were both performed.")
    print()

    print("Conclusion:")
    print("  See the numerical comparison above to determine whether")
    print("  device-dependent numerical differences are large enough")
    print("  to explain the discrepancy.")

else:

    print("Result:")
    print("  GPU was unavailable in this environment.")
    print()

    print("Conclusion:")
    print("  CPU vs GPU differences were NOT tested.")
    print("  This remains an open possibility.")


TEST 8 — CPU vs GPU INFERENCE
Input shape: torch.Size([100, 3, 152, 152])
Input dtype: torch.float32

Running CPU inference...
Running GPU inference...

TEST 8 — CPU vs GPU EMBEDDING COMPARISON
----------------------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         False
Max absolute diff:     2.7562141418e-02
Mean absolute diff:    1.1923504644e-04
RMSE:                  5.6956283515e-04

TEST 8 — SUMMARY
Question:
Can CPU vs GPU inference explain the embedding discrepancy?

Result:
  CPU and GPU inference were both performed.

Conclusion:
  See the numerical comparison above to determine whether
  device-dependent numerical differences are large enough
  to explain the discrepancy.


In [18]:
# ============================================================
# TEST 9 — DIRECT OFFICIAL HDF5 REPRODUCTION
# ============================================================
#
# QUESTION
# --------
# Can we reproduce the official SGA embeddings by taking the
# images directly from the exact HDF5 chunk used to generate
# them and passing those images directly through the official
# backbone-loading procedure?
#
# PURPOSE
# -------
# This isolates the official embedding pipeline from our
# DecalsDataset entirely.
#
# OFFICIAL PIPELINE (from extract_embeddings)
# --------------------------------------------
#
#     HDF5["images"]
#          |
#          v
#     torch.from_numpy()
#          |
#          v
#     backbone(batch)
#          |
#          v
#     2048-D embedding
#
# No DecalsDataset is involved.
#
# If this reproduces the official embeddings, then the
# discrepancy comes from our image-loading / transformation
# pipeline.
#
# If this DOES NOT reproduce the official embeddings, then
# something about the official embedding-generation process
# remains different from extract_embeddings() as currently
# understood.
#
# ============================================================

print()
print("=" * 60)
print("TEST 9 — DIRECT OFFICIAL HDF5 REPRODUCTION")
print("=" * 60)


N_TEST = len(test_ids)


# ------------------------------------------------------------
# Load the exact official input images
# ------------------------------------------------------------

with h5py.File(CHUNK_PATH, "r") as f:

    official_raw_images = f["images"][:N_TEST]
    official_raw_ids = f["sgaid"][:N_TEST]


# ------------------------------------------------------------
# Verify galaxy identity
# ------------------------------------------------------------

if not np.array_equal(
    official_raw_ids,
    test_ids
):

    raise ValueError(
        "Official HDF5 SGAIDs do not match test_ids!"
    )

print("SGAIDs verified.")


# ------------------------------------------------------------
# Inspect official HDF5 image representation
# ------------------------------------------------------------

print()
print("OFFICIAL HDF5 INPUT")
print("-------------------")

print(
    "Shape:",
    official_raw_images.shape
)

print(
    "dtype:",
    official_raw_images.dtype
)

print(
    "min:",
    official_raw_images.min()
)

print(
    "max:",
    official_raw_images.max()
)

print(
    "mean:",
    official_raw_images.mean()
)

print(
    "std:",
    official_raw_images.std()
)


# ------------------------------------------------------------
# Convert exactly as extract_embeddings() does
# ------------------------------------------------------------
#
# The official code does:
#
#     batch = torch.from_numpy(images[start:end])
#
# Therefore we intentionally perform NO transpose,
# normalization, augmentation, or other transformation here.
#
# ------------------------------------------------------------

official_input = torch.from_numpy(
    official_raw_images
)

print()
print("DIRECT OFFICIAL INPUT")
print("---------------------")

print(
    "Shape:",
    official_input.shape
)

print(
    "dtype:",
    official_input.dtype
)


# ------------------------------------------------------------
# Load official backbone using the same implementation
# used by extract_embeddings().
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

official_direct_backbone = load_official_backbone(
    OFFICIAL_CHECKPOINT,
    device
)


# ------------------------------------------------------------
# Generate embeddings
# ------------------------------------------------------------

print()
print("Generating embeddings directly from official HDF5...")

official_reproduced = []

with torch.no_grad():

    for start in tqdm(
        range(
            0,
            len(official_input),
            EMBEDDING_BATCH_SIZE
        ),
        desc="Official HDF5 inference"
    ):

        end = min(
            start + EMBEDDING_BATCH_SIZE,
            len(official_input)
        )

        batch = official_input[start:end].to(device)

        output = official_direct_backbone(batch)

        official_reproduced.append(
            output.cpu()
        )

official_reproduced = torch.cat(
    official_reproduced
)


# ------------------------------------------------------------
# Convert stored official embeddings to torch
# ------------------------------------------------------------

official_stored = torch.from_numpy(
    official_embeddings
)


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

compare_tensors(
    official_reproduced,
    official_stored,
    "TEST 9 — DIRECT HDF5 REPRODUCED vs OFFICIAL"
)

# ============================================================
# TEST 9 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 9 — SUMMARY")
print("=" * 60)

print("Question:")
print("Can the official embeddings be reproduced directly from")
print("the official HDF5 images using the official checkpoint?")
print()

print("Result:")
print(f"  SGAIDs verified:        {np.array_equal(official_raw_ids, test_ids)}")
print(f"  Exactly equal:          {torch.equal(official_reproduced, official_stored)}")
print(f"  Max absolute diff:      {torch.max(torch.abs(official_reproduced - official_stored)).item():.10e}")
print(f"  Mean absolute diff:     {torch.mean(torch.abs(official_reproduced - official_stored)).item():.10e}")
print(f"  RMSE:                   {torch.sqrt(torch.mean((official_reproduced - official_stored) ** 2)).item():.10e}")
print()

print("Conclusion:")
print("  Even when the exact official HDF5 images are passed directly")
print("  through the official checkpoint/backbone implementation,")
print("  the stored official embeddings are not reproduced exactly.")
print()
print("  Therefore, the discrepancy is NOT caused by DecalsDataset")
print("  or by the choice of galaxies/images used in our comparison.")
print()
print("  The historical official embedding-generation environment")
print("  or an additional difference in the original extraction")
print("  pipeline remains to be investigated.")


TEST 9 — DIRECT OFFICIAL HDF5 REPRODUCTION
SGAIDs verified.

OFFICIAL HDF5 INPUT
-------------------
Shape: (100, 3, 152, 152)
dtype: float32
min: -2.5995345
max: 44.55522
mean: 0.016296143
std: 0.2560547

DIRECT OFFICIAL INPUT
---------------------
Shape: torch.Size([100, 3, 152, 152])
dtype: torch.float32
Missing keys: []
Unexpected keys: ['fc.0.weight', 'fc.0.bias', 'fc.2.weight', 'fc.2.bias']

Generating embeddings directly from official HDF5...


Official HDF5 inference: 100%|██████████| 1/1 [00:00<00:00, 39.67it/s]


TEST 9 — DIRECT HDF5 REPRODUCED vs OFFICIAL
-------------------------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         False
Max absolute diff:     1.3046264648e-02
Mean absolute diff:    7.3706658441e-05
RMSE:                  3.1403396861e-04

TEST 9 — SUMMARY
Question:
Can the official embeddings be reproduced directly from
the official HDF5 images using the official checkpoint?

Result:
  SGAIDs verified:        True
  Exactly equal:          False
  Max absolute diff:      1.3046264648e-02
  Mean absolute diff:     7.3706658441e-05
  RMSE:                   3.1403396861e-04

Conclusion:
  Even when the exact official HDF5 images are passed directly
  through the official checkpoint/backbone implementation,
  the stored official embeddings are not reproduced exactly.

  Therefore, the discrepancy is NOT caused by DecalsDataset
  or by the choice of galaxies/images used in our comparison.

  The historical official embedding-generation environ

In [19]:
# ============================================================
# TEST 10 — EMBEDDING COSINE SIMILARITY
# ============================================================
#
# QUESTION
# --------
# Although the reproduced and official embeddings are not
# numerically identical, do they point in essentially the
# same direction in the 2048-dimensional embedding space?
#
# PURPOSE
# -------
# Direct element-by-element equality is a very strict test.
# For many embedding-based applications, the relative direction
# of vectors is more important than tiny numerical differences.
#
# We therefore calculate:
#
#   1. Cosine similarity between each reproduced/official pair
#   2. Euclidean difference between the vectors
#   3. Norm of each vector
#   4. Relative norm difference
#
# INTERPRETATION
# --------------
# Cosine similarity near 1.0 means the vectors point in almost
# exactly the same direction even if individual floating-point
# values differ.
#
# This test does NOT establish that the two embedding pipelines
# are scientifically interchangeable. It tells us whether the
# observed discrepancy is primarily a small numerical difference
# or a substantial change in embedding direction.
#
# ============================================================

print()
print("=" * 60)
print("TEST 10 — EMBEDDING COSINE SIMILARITY")
print("=" * 60)


# ------------------------------------------------------------
# Convert to NumPy
# ------------------------------------------------------------

reproduced = official_reproduced.numpy()
stored = official_stored.numpy()


# ------------------------------------------------------------
# Calculate vector norms
# ------------------------------------------------------------

reproduced_norms = np.linalg.norm(
    reproduced,
    axis=1
)

stored_norms = np.linalg.norm(
    stored,
    axis=1
)


# ------------------------------------------------------------
# Calculate cosine similarity
#
# cosine(a,b) = (a · b) / (||a|| ||b||)
# ------------------------------------------------------------

dot_products = np.sum(
    reproduced * stored,
    axis=1
)

cosine_similarities = (
    dot_products /
    (reproduced_norms * stored_norms)
)


# ------------------------------------------------------------
# Euclidean distances
# ------------------------------------------------------------

euclidean_distances = np.linalg.norm(
    reproduced - stored,
    axis=1
)


# ------------------------------------------------------------
# Relative norm differences
# ------------------------------------------------------------

relative_norm_difference = (
    np.abs(reproduced_norms - stored_norms)
    / stored_norms
)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print()
print("COSINE SIMILARITY")
print("-----------------")

print(
    f"Minimum:       {np.min(cosine_similarities):.12f}"
)

print(
    f"Maximum:       {np.max(cosine_similarities):.12f}"
)

print(
    f"Mean:          {np.mean(cosine_similarities):.12f}"
)

print(
    f"Median:        {np.median(cosine_similarities):.12f}"
)

print(
    f"Std:           {np.std(cosine_similarities):.12e}"
)


print()
print("EUCLIDEAN DISTANCE")
print("------------------")

print(
    f"Minimum:       {np.min(euclidean_distances):.10e}"
)

print(
    f"Maximum:       {np.max(euclidean_distances):.10e}"
)

print(
    f"Mean:          {np.mean(euclidean_distances):.10e}"
)

print(
    f"Median:        {np.median(euclidean_distances):.10e}"
)


print()
print("EMBEDDING NORMS")
print("---------------")

print(
    f"Reproduced mean norm:   "
    f"{np.mean(reproduced_norms):.10e}"
)

print(
    f"Official mean norm:     "
    f"{np.mean(stored_norms):.10e}"
)


print()
print("RELATIVE NORM DIFFERENCE")
print("------------------------")

print(
    f"Minimum:       "
    f"{np.min(relative_norm_difference):.10e}"
)

print(
    f"Maximum:       "
    f"{np.max(relative_norm_difference):.10e}"
)

print(
    f"Mean:          "
    f"{np.mean(relative_norm_difference):.10e}"
)


# ------------------------------------------------------------
# Per-galaxy table for the first 10 objects
# ------------------------------------------------------------

print()
print("FIRST 10 GALAXIES")
print("-----------------")

print(
    f"{'SGAID':>12} "
    f"{'Cosine':>16} "
    f"{'Euclidean':>16} "
    f"{'Rel. Norm Diff':>18}"
)

for i in range(min(10, len(test_ids))):

    print(
        f"{test_ids[i]:12d} "
        f"{cosine_similarities[i]:16.12f} "
        f"{euclidean_distances[i]:16.8e} "
        f"{relative_norm_difference[i]:18.8e}"
    )


# ============================================================
# TEST 10 — SUMMARY
# ============================================================

print()
print("=" * 60)
print("TEST 10 — SUMMARY")
print("=" * 60)

print("Question:")
print("Are the reproduced and official embeddings nearly identical")
print("in direction despite their element-wise numerical differences?")
print()

print(
    f"Mean cosine similarity:    "
    f"{np.mean(cosine_similarities):.12f}"
)

print(
    f"Minimum cosine similarity: "
    f"{np.min(cosine_similarities):.12f}"
)

print(
    f"Mean Euclidean distance:    "
    f"{np.mean(euclidean_distances):.10e}"
)

print(
    f"Mean relative norm diff:    "
    f"{np.mean(relative_norm_difference):.10e}"
)

print()

if np.min(cosine_similarities) > 0.9999:

    print("Conclusion:")
    print("  The reproduced and official embeddings have extremely")
    print("  similar directions despite not being element-wise equal.")
    print("  This strongly suggests that the discrepancy is relatively")
    print("  small in the geometry of the embedding space.")

else:

    print("Conclusion:")
    print("  The embedding directions show meaningful differences.")
    print("  Further investigation of the official pipeline is warranted.")


TEST 10 — EMBEDDING COSINE SIMILARITY

COSINE SIMILARITY
-----------------
Minimum:       0.999999761581
Maximum:       1.000000119209
Mean:          1.000000000000
Median:        0.999999940395
Std:           1.211309523796e-07

EUCLIDEAN DISTANCE
------------------
Minimum:       3.4944672370e-04
Maximum:       7.1720175445e-02
Mean:          6.3213054091e-03
Median:        1.1513642967e-03

EMBEDDING NORMS
---------------
Reproduced mean norm:   2.2284145355e+01
Official mean norm:     2.2284212112e+01

RELATIVE NORM DIFFERENCE
------------------------
Minimum:       3.2957707390e-06
Maximum:       8.5284933448e-04
Mean:          2.3543271527e-04

FIRST 10 GALAXIES
-----------------
       SGAID           Cosine        Euclidean     Rel. Norm Diff
     4976559   0.999999821186   7.77193811e-04     1.71646825e-04
     4976561   1.000000119209   8.27699550e-04     3.05410213e-04
     4976562   0.999999821186   7.05188955e-04     3.51231196e-04
     4976563   0.999999880791   4.395160

In [33]:
# ============================================================
# TEST 11 — SOFTWARE / COMPUTATIONAL ENVIRONMENT
# ============================================================
#
# QUESTION
# --------
# Are there software or hardware environment differences that
# could plausibly affect numerical reproducibility?
#
# PURPOSE
# -------
# Record the computational environment used to reproduce the
# official embeddings. This does not by itself identify a cause,
# but establishes the environment for reproducibility.
#
# ============================================================

import sys
import platform
import torch
import torchvision
import numpy as np
import h5py

print()
print("=" * 60)
print("TEST 11 — SOFTWARE / COMPUTATIONAL ENVIRONMENT")
print("=" * 60)

print()
print("PYTHON")
print("------")
print("Version:", sys.version)

print()
print("OPERATING SYSTEM")
print("-----------------")
print("System:       ", platform.system())
print("Release:      ", platform.release())
print("Machine:      ", platform.machine())

print()
print("PYTORCH")
print("-------")
print("Version:      ", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version: ", torch.version.cuda)

print()
print("TORCHVISION")
print("-----------")
print("Version:", torchvision.__version__)

print()
print("NUMPY")
print("-----")
print("Version:", np.__version__)

print()
print("H5PY")
print("----")
print("Version:", h5py.__version__)

if torch.cuda.is_available():

    print()
    print("GPU")
    print("---")

    print("Device:", torch.cuda.get_device_name(0))
    print("Device count:", torch.cuda.device_count())

else:

    print()
    print("GPU")
    print("---")
    print("No CUDA GPU available.")

print()
print("PYTORCH DETERMINISM SETTINGS")
print("----------------------------")
print(
    "torch.backends.cudnn.deterministic:",
    torch.backends.cudnn.deterministic
)
print(
    "torch.backends.cudnn.benchmark:",
    torch.backends.cudnn.benchmark
)


TEST 11 — SOFTWARE / COMPUTATIONAL ENVIRONMENT

PYTHON
------
Version: 3.9.9 | packaged by conda-forge | (main, Dec 20 2021, 02:41:03) 
[GCC 9.4.0]

OPERATING SYSTEM
-----------------
System:        Linux
Release:       6.4.0-150600.23.125_15.0.27-cray_shasta_c
Machine:       x86_64

PYTORCH
-------
Version:       1.10.1
CUDA available: False
CUDA version:  11.3

TORCHVISION
-----------
Version: 0.11.2+cu102

NUMPY
-----
Version: 1.21.6

H5PY
----
Version: 3.6.0

GPU
---
No CUDA GPU available.

PYTORCH DETERMINISM SETTINGS
----------------------------
torch.backends.cudnn.deterministic: False
torch.backends.cudnn.benchmark: False


In [34]:
# ============================================================
# TEST 12 — BATCH-SIZE DEPENDENCE
# ============================================================
#
# QUESTION
# --------
# Does changing the inference batch size change the resulting
# ResNet50 embeddings?
#
# PURPOSE
# -------
# Determine whether BatchNorm, numerical execution, or other
# batch-dependent behavior can affect the backbone embeddings.
#
# The input images and model weights are held constant.
#
# ============================================================

print()
print("=" * 60)
print("TEST 12 — BATCH-SIZE DEPENDENCE")
print("=" * 60)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BATCH_SIZES = [1, 8, 16, 32, 64, 100]


# ------------------------------------------------------------
# Use the official HDF5 input from Test 9
# ------------------------------------------------------------

test_input = raw_images_torch.to(device)

print()
print("Input shape:", test_input.shape)
print("Input dtype:", test_input.dtype)
print("Model mode:", official_backbone.training)


# ------------------------------------------------------------
# Generate embeddings for each batch size
# ------------------------------------------------------------

batch_embeddings = {}


for batch_size in BATCH_SIZES:

    print()
    print(f"Running batch size = {batch_size}")

    outputs = []

    with torch.no_grad():

        for start in range(
            0,
            len(test_input),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(test_input)
            )

            output = official_backbone(
                test_input[start:end]
            )

            outputs.append(
                output.cpu()
            )

    batch_embeddings[batch_size] = torch.cat(outputs)


# ------------------------------------------------------------
# Compare every batch size against batch size 1
# ------------------------------------------------------------

reference = batch_embeddings[1]

print()
print("=" * 60)
print("TEST 12 — RESULTS")
print("=" * 60)

print()
print(
    f"{'Batch Size':>12} "
    f"{'Exact':>10} "
    f"{'Max Diff':>16} "
    f"{'Mean Diff':>16} "
    f"{'RMSE':>16}"
)

print("-" * 76)


for batch_size in BATCH_SIZES:

    current = batch_embeddings[batch_size]

    difference = current - reference

    exact = torch.equal(
        current,
        reference
    )

    max_diff = torch.max(
        torch.abs(difference)
    ).item()

    mean_diff = torch.mean(
        torch.abs(difference)
    ).item()

    rmse = torch.sqrt(
        torch.mean(difference ** 2)
    ).item()

    print(
        f"{batch_size:>12} "
        f"{str(exact):>10} "
        f"{max_diff:>16.10e} "
        f"{mean_diff:>16.10e} "
        f"{rmse:>16.10e}"
    )


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print()
print("INTERPRETATION")
print("--------------")

all_identical = all(
    torch.equal(
        batch_embeddings[1],
        batch_embeddings[b]
    )
    for b in BATCH_SIZES
)

if all_identical:

    print(
        "All tested batch sizes produced exactly identical "
        "embeddings."
    )

    print(
        "Batch-size-dependent numerical behavior is therefore "
        "not responsible for the discrepancy observed in "
        "Tests 7 and 9."
    )

else:

    print(
        "At least one batch size produced different embeddings."
    )

    print(
        "Batch-size-dependent computation may contribute to "
        "the discrepancy and requires further investigation."
    )


TEST 12 — BATCH-SIZE DEPENDENCE

Input shape: torch.Size([100, 3, 152, 152])
Input dtype: torch.float32
Model mode: False

Running batch size = 1

Running batch size = 8

Running batch size = 16

Running batch size = 32

Running batch size = 64

Running batch size = 100

TEST 12 — RESULTS

  Batch Size      Exact         Max Diff        Mean Diff             RMSE
----------------------------------------------------------------------------
           1       True 0.0000000000e+00 0.0000000000e+00 0.0000000000e+00
           8      False 1.3351440430e-05 6.9749916065e-08 3.6070480292e-07
          16      False 1.3351440430e-05 6.9749916065e-08 3.6070480292e-07
          32      False 1.3351440430e-05 6.9749916065e-08 3.6070480292e-07
          64      False 1.3351440430e-05 6.9749916065e-08 3.6070480292e-07
         100      False 1.3351440430e-05 6.9749916065e-08 3.6070480292e-07

INTERPRETATION
--------------
At least one batch size produced different embeddings.
Batch-size-dependent

In [35]:
# ============================================================
# TEST 13 — EVAL MODE / BATCHNORM BEHAVIOR
# ============================================================
#
# QUESTION
# --------
# Does the model's training/evaluation state affect the
# resulting backbone embeddings?
#
# PURPOSE
# -------
# Determine whether BatchNorm or other train/eval-dependent
# behavior could explain the difference between reproduced and
# official embeddings.
#
# IMPORTANT
# ---------
# The official extraction function is expected to use eval mode.
# The train-mode experiment is therefore diagnostic only.
#
# ============================================================

print()
print("=" * 60)
print("TEST 13 — EVAL MODE / BATCHNORM BEHAVIOR")
print("=" * 60)


# ------------------------------------------------------------
# Inspect BatchNorm layers
# ------------------------------------------------------------

batchnorm_layers = []

for name, module in official_backbone.named_modules():

    if isinstance(
        module,
        torch.nn.modules.batchnorm._BatchNorm
    ):

        batchnorm_layers.append(
            (name, module)
        )


print()
print("BATCHNORM LAYERS")
print("----------------")

print(
    "Number of BatchNorm layers:",
    len(batchnorm_layers)
)

for name, module in batchnorm_layers:

    print(
        f"{name}: "
        f"training={module.training}, "
        f"num_features={module.num_features}"
    )


# ------------------------------------------------------------
# Fixed input
# ------------------------------------------------------------

test_input = raw_images_torch.to(device)


# ------------------------------------------------------------
# EVAL MODE
# ------------------------------------------------------------

official_backbone.eval()

print()
print("Running EVAL mode...")

with torch.no_grad():

    eval_output = official_backbone(
        test_input
    ).cpu()


# ------------------------------------------------------------
# TRAIN MODE
# ------------------------------------------------------------

official_backbone.train()

print("Running TRAIN mode...")

with torch.no_grad():

    train_output = official_backbone(
        test_input
    ).cpu()


# ------------------------------------------------------------
# Restore EVAL MODE
# ------------------------------------------------------------

official_backbone.eval()


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

compare_tensors(
    eval_output,
    train_output,
    "TEST 13 — EVAL vs TRAIN OUTPUT"
)


# ------------------------------------------------------------
# Compare BatchNorm running statistics
# ------------------------------------------------------------

print()
print("BATCHNORM STATE AFTER TRAIN-MODE INFERENCE")
print("-------------------------------------------")

print(
    "Note: train-mode inference updates BatchNorm running "
    "statistics even under torch.no_grad()."
)

print(
    "The model has been restored to eval mode, but the "
    "BatchNorm statistics may now differ from their original "
    "values."
)


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print()
print("INTERPRETATION")
print("--------------")

print(
    "The official embedding extraction procedure should use "
    "eval mode."
)

print(
    "Therefore, any difference between train and eval mode "
    "does not indicate that train mode was used officially; "
    "it establishes whether model state can materially affect "
    "the embeddings."
)


TEST 13 — EVAL MODE / BATCHNORM BEHAVIOR

BATCHNORM LAYERS
----------------
Number of BatchNorm layers: 53
bn1: training=False, num_features=64
layer1.0.bn1: training=False, num_features=64
layer1.0.bn2: training=False, num_features=64
layer1.0.bn3: training=False, num_features=256
layer1.0.downsample.1: training=False, num_features=256
layer1.1.bn1: training=False, num_features=64
layer1.1.bn2: training=False, num_features=64
layer1.1.bn3: training=False, num_features=256
layer1.2.bn1: training=False, num_features=64
layer1.2.bn2: training=False, num_features=64
layer1.2.bn3: training=False, num_features=256
layer2.0.bn1: training=False, num_features=128
layer2.0.bn2: training=False, num_features=128
layer2.0.bn3: training=False, num_features=512
layer2.0.downsample.1: training=False, num_features=512
layer2.1.bn1: training=False, num_features=128
layer2.1.bn2: training=False, num_features=128
layer2.1.bn3: training=False, num_features=512
layer2.2.bn1: training=False, num_features=1

In [36]:
# ============================================================
# TEST 14 — FLOATING-POINT PRECISION
# ============================================================
#
# QUESTION
# --------
# Could differences in floating-point precision account for
# the small numerical differences between reproduced and
# official embeddings?
#
# PURPOSE
# -------
# Determine whether the embedding discrepancy changes when
# inference is performed at different floating-point
# precisions.
#
# IMPORTANT
# ---------
# TEST 13 ran the model in TRAIN mode, which modified the
# BatchNorm running statistics.
#
# Therefore, the model is RELOADED from the checkpoint here
# before performing Test 14.
#
# This ensures Test 14 starts from the original checkpoint
# state rather than the modified BatchNorm state from Test 13.
#
# ============================================================

print()
print("=" * 60)
print("TEST 14 — FLOATING-POINT PRECISION")
print("=" * 60)


# ------------------------------------------------------------
# Reload model after TEST 13
# ------------------------------------------------------------

print()
print("Reloading model after TEST 13...")
print("Reason: TEST 13 modified BatchNorm running statistics.")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Load a completely fresh copy from the checkpoint
precision_model = Moco_v2.load_from_checkpoint(
    checkpoint_path=MY_CHECKPOINT
).encoder_q

precision_model.fc = torch.nn.Identity()

precision_model = precision_model.to(device).eval()


print("Model reloaded.")
print("Model mode:", precision_model.training)


# ------------------------------------------------------------
# Verify BatchNorm layers are in eval mode
# ------------------------------------------------------------

bn_layers = [
    module
    for module in precision_model.modules()
    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm)
]

print("BatchNorm layers:", len(bn_layers))
print(
    "All BatchNorm layers in eval mode:",
    all(not bn.training for bn in bn_layers)
)


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def run_precision_inference(model, images, dtype):
    """
    Run inference using the requested floating-point dtype.

    Returns float32 embeddings so that results from different
    precision modes can be compared directly.
    """

    model = model.to(device)

    images_device = images.to(
        device=device,
        dtype=dtype
    )

    with torch.no_grad():

        output = model(images_device)

    return output.float().cpu()


# ------------------------------------------------------------
# FLOAT32 BASELINE
# ------------------------------------------------------------

print()
print("Running FLOAT32 inference...")

embeddings_float32 = run_precision_inference(
    precision_model,
    processed,
    torch.float32
)


# ------------------------------------------------------------
# FLOAT64
# ------------------------------------------------------------

print()
print("Running FLOAT64 inference...")

try:

    precision_model_double = Moco_v2.load_from_checkpoint(
        checkpoint_path=MY_CHECKPOINT
    ).encoder_q

    precision_model_double.fc = torch.nn.Identity()

    precision_model_double = (
        precision_model_double
        .double()
        .to(device)
        .eval()
    )

    embeddings_float64 = run_precision_inference(
        precision_model_double,
        processed,
        torch.float64
    )

    float64_available = True

except Exception as e:

    print("FLOAT64 inference failed:")
    print(e)

    embeddings_float64 = None
    float64_available = False


# ------------------------------------------------------------
# FLOAT32 vs FLOAT64
# ------------------------------------------------------------

if float64_available:

    compare_tensors(
        embeddings_float32,
        embeddings_float64,
        "TEST 14a — FLOAT32 vs FLOAT64"
    )


# ------------------------------------------------------------
# FLOAT16
# ------------------------------------------------------------

print()
print("Running FLOAT16 inference...")

try:

    precision_model_half = Moco_v2.load_from_checkpoint(
        checkpoint_path=MY_CHECKPOINT
    ).encoder_q

    precision_model_half.fc = torch.nn.Identity()

    precision_model_half = (
        precision_model_half
        .half()
        .to(device)
        .eval()
    )

    embeddings_float16 = run_precision_inference(
        precision_model_half,
        processed,
        torch.float16
    )

    float16_available = True

except Exception as e:

    print("FLOAT16 inference failed:")
    print(e)

    embeddings_float16 = None
    float16_available = False


# ------------------------------------------------------------
# FLOAT32 vs FLOAT16
# ------------------------------------------------------------

if float16_available:

    compare_tensors(
        embeddings_float32,
        embeddings_float16,
        "TEST 14b — FLOAT32 vs FLOAT16"
    )


# ------------------------------------------------------------
# Compare FLOAT32 reproduction to official embeddings
# ------------------------------------------------------------

print()
print("=" * 60)
print("TEST 14 — FLOATING-POINT PRECISION SUMMARY")
print("=" * 60)

print()
print("FLOAT32 reproduction vs official")
print("----------------------------------")

float32_official_diff = (
    embeddings_float32.numpy()
    - official_embeddings
)

print(
    f"Max absolute difference: "
    f"{np.max(np.abs(float32_official_diff)):.10e}"
)

print(
    f"Mean absolute difference: "
    f"{np.mean(np.abs(float32_official_diff)):.10e}"
)

print(
    f"RMSE: "
    f"{np.sqrt(np.mean(float32_official_diff ** 2)):.10e}"
)

print(
    "Exactly equal:",
    np.array_equal(
        embeddings_float32.numpy(),
        official_embeddings
    )
)


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print()
print("INTERPRETATION")
print("--------------")

print(
    """
TEST 14 establishes that changing numerical precision from
FLOAT32 to FLOAT64 produces only very small changes in the
2048-dimensional backbone embeddings.

The FLOAT32-vs-FLOAT64 RMSE is much smaller than the
reproduced-vs-official RMSE observed in TEST 9. Therefore,
floating-point precision is unlikely to be the primary cause
of the discrepancy between our reproduced embeddings and the
official SGA embeddings.

FLOAT16 inference was not available on this CPU build and
therefore was not evaluated.

NOTE:
The reproduced-vs-official comparison printed above uses
`processed` (our DecalsDataset output), not the official HDF5
input. The direct official-HDF5 comparison from TEST 9 is the
appropriate comparison for isolating model/pipeline
differences.
"""
)


TEST 14 — FLOATING-POINT PRECISION

Reloading model after TEST 13...
Reason: TEST 13 modified BatchNorm running statistics.
Model reloaded.
Model mode: False
BatchNorm layers: 53
All BatchNorm layers in eval mode: True

Running FLOAT32 inference...

Running FLOAT64 inference...

TEST 14a — FLOAT32 vs FLOAT64
-----------------------------
Shape:                 torch.Size([100, 2048])
Exactly equal:         False
Max absolute diff:     3.0517578125e-05
Mean absolute diff:    1.1024757640e-07
RMSE:                  5.4821367712e-07

Running FLOAT16 inference...
FLOAT16 inference failed:
"unfolded2d_copy" not implemented for 'Half'

TEST 14 — FLOATING-POINT PRECISION SUMMARY

FLOAT32 reproduction vs official
----------------------------------
Max absolute difference: 3.4445152283e+00
Mean absolute difference: 1.7462732270e-02
RMSE: 8.5430189967e-02
Exactly equal: False

INTERPRETATION
--------------

This test asks whether changing floating-point precision
produces differences comparable

# Findings So Far

| Test | Question | Result | Implication |
|---|---|---|---|
| 1 | Are `rrjc` augmentations stochastic? | YES | Same galaxy can produce different embeddings on different passes |
| 2 | Is inference deterministic without augmentation? | YES | Neural-network inference itself is deterministic |
| 3 | Do deterministic embeddings reproduce SGA? | NO | Augmentation alone does not explain the SGA discrepancy |
| 4 | Are the checkpoints identical? | YES | Checkpoint weights are not the cause |
| 5 | Does our no-augmentation pipeline reproduce the raw HDF5 input? | TBD | Tests preprocessing/input differences |
| 6 | Do the two model-loading methods produce identical outputs? | TBD | Tests model construction/loading differences |

## Current interpretation

Our morphology-classification pipeline and the official SGA2025
pipeline use the same underlying checkpoint, but our embeddings
do not exactly reproduce the official embeddings.

The old morphology-classification pipeline used `rrjc`
(RandomRotate + JitterCrop), which introduces stochasticity.
However, disabling those augmentations makes the pipeline
completely deterministic and still does not reproduce the
official SGA embeddings.

Therefore, augmentation randomness is a real source of
embedding variation in the old classifier, but it is not by
itself sufficient to explain the difference between our
embeddings and SGA's.

Further investigation is focused on differences between the
official embedding extraction pipeline and our preprocessing
/model-loading pipeline.